# Notebook 11 -- Executive Summary

## 1. Load Data

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Brand-Level Summary

In [2]:
df = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")

scored_brands = ["Starbucks", "Dunkin'", "Independent Café"]
rows = []
for b in scored_brands:
    g = df[df["brand_category"] == b]
    elite = g[g["user_activity_tier"] == "Elite"]["review_stars"].mean()
    rows.append({
        "Brand":           b,
        "Reviews":         f"{len(g):,}",
        "Avg Stars":       f"{g['review_stars'].mean():.2f}",
        "% Positive":      f"{(g['star_tier']=='Positive').mean()*100:.1f}%",
        "% Critical":      f"{(g['star_tier']=='Critical').mean()*100:.1f}%",
        "Avg Sentiment":   f"{g['sentiment_score'].mean():.3f}",
        "Elite Avg Stars": f"{elite:.2f}",
    })

summary = pd.DataFrame(rows)
display(summary)

,Brand,Reviews,Avg Stars,% Positive,% Critical,Avg Sentiment,Elite Avg Stars
0,Starbucks,"11,675",2.90,42.4%,47.6%,0.277,3.60
1,Dunkin',"6,866",2.05,21.5%,71.2%,-0.020,3.01
2,Independent Café,"362,334",4.01,73.7%,17.2%,0.706,4.10


## 3. Topic Scorecard

In [3]:
topic_dims = ["Service", "Food Quality", "Ambiance", "Wait Time", "Cleanliness"]

scores = {}
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rates = {}
    for t in topic_dims:
        # Multi-label: count reviews that mention this topic (may also carry other tags)
        mask  = g["topic_tags"].str.contains(t, regex=False)
        total = mask.sum()
        pos   = (mask & (g["star_tier"] == "Positive")).sum()
        rates[t] = round(pos / total * 100, 1) if total > 0 else 0
    scores[b] = rates

score_df = pd.DataFrame(scores).T
score_df["Composite"] = score_df.mean(axis=1).round(1)
display(score_df)

theta = topic_dims + [topic_dims[0]]
brand_styles = {
    "Starbucks":        {"color": "#00704A", "dash": "solid"},
    "Dunkin'":          {"color": "#d62728", "dash": "dash"},
    "Independent Café": {"color": "#aaaaaa", "dash": "dot"},
}

fig = go.Figure()
for brand, style in brand_styles.items():
    vals = [scores[brand][d] for d in topic_dims] + [scores[brand][topic_dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=theta, name=brand,
        mode="lines+markers",
        line=dict(color=style["color"], dash=style["dash"], width=2.5),
        marker=dict(color=style["color"], size=8),
        fill="toself", fillcolor=style["color"], opacity=0.12,
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True, range=[0, 100],
            tickvals=[0, 25, 50, 75, 100], ticksuffix="%",
            gridcolor="#dddddd", linecolor="#dddddd",
        ),
        angularaxis=dict(gridcolor="#dddddd", linecolor="#dddddd"),
    ),
    title=dict(text="Brand Scorecard — Topic Positive Rate (%)", font=dict(size=16), x=0.5),
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    paper_bgcolor="white",
    width=620, height=580,
    margin=dict(t=80, b=90, l=60, r=60),
)
fig.write_html(str(FIGURES_DIR / "11_scorecard_summary.html"))
fig.show()

,Service,Food Quality,Ambiance,Wait Time,Cleanliness,Composite
Starbucks,44.4,39.5,62.2,39.9,47.2,46.6
Dunkin',22.1,22.2,42.2,25.2,27.2,27.8
Independent Café,71.5,73.9,78.8,65.4,62.4,70.4


## Key Findings

- Starbucks averages 2.90 stars and 42.4% positive. Below independents (4.01 stars, 73.7% positive), above Dunkin' (2.05 stars, 21.5% positive).
- 71.2% of Dunkin' reviews are critical, with average sentiment of -0.020.
- Independents lead every topic dimension. Composite score: 70.4 vs. Starbucks 46.6 and Dunkin' 27.8. The widest gap is Food Quality (73.9% vs. 39.5% vs. 22.2%).
- Starbucks Elite reviewers average 3.60 stars vs. the overall 2.90, suggesting higher-engagement customers have better experiences.